In [2]:
import sys
from pathlib import Path

sys.path.append(str(Path.cwd().parents[1]))

from dashboard.database import load_dashboard_data

df = load_dashboard_data()

df.shape

(174724, 37)

In [3]:
departures = (
    df.dropna(subset=["start_station_name"])
      .groupby("start_station_name")
      .size()
      .rename("departures")
)

In [4]:
arrivals = (
    df.dropna(subset=["end_station_name"])
      .groupby("end_station_name")
      .size()
      .rename("arrivals")
)

In [5]:
all_stations = sorted(
    set(departures.index) | set(arrivals.index)
)

In [6]:
start_coords = (
    df.dropna(subset=["start_station_name"])
      .groupby("start_station_name")[
          ["start_latitude", "start_longitude"]
      ]
      .first()
      .rename(columns={
          "start_latitude": "lat",
          "start_longitude": "lon"
      })
)

In [7]:
end_coords = (
    df.dropna(subset=["end_station_name"])
      .groupby("end_station_name")[
          ["end_latitude", "end_longitude"]
      ]
      .first()
      .rename(columns={
          "end_latitude": "lat",
          "end_longitude": "lon"
      })
)

In [8]:
coords = start_coords.combine_first(end_coords)

In [12]:
import pandas as pd


station_metrics = pd.DataFrame(
    index=all_stations
)

station_metrics.index.name = "station_name"

station_metrics = station_metrics.join(departures)
station_metrics = station_metrics.join(arrivals)
station_metrics = station_metrics.join(coords)

station_metrics = station_metrics.fillna({
    "departures": 0,
    "arrivals": 0
})

station_metrics["total_traffic"] = (
    station_metrics["departures"] +
    station_metrics["arrivals"]
)

station_metrics["net_flow"] = (
    station_metrics["arrivals"] -
    station_metrics["departures"]
)

station_metrics["imbalance_ratio"] = (
    station_metrics["net_flow"] /
    station_metrics["total_traffic"]
    * 100
)

station_metrics["imbalance_ratio"] = (
    station_metrics["imbalance_ratio"]
    .replace([float("inf"), -float("inf")], 0)
    .fillna(0)
)

station_metrics["has_valid_coords"] = (
    station_metrics["lat"].notna() &
    station_metrics["lon"].notna()
)

station_metrics = (
    station_metrics
    .reset_index()
    .sort_values("total_traffic", ascending=False)
    .reset_index(drop=True)
)

In [13]:
station_metrics.head(10)

,station_name,departures,arrivals,lat,lon,total_traffic,net_flow,imbalance_ratio,has_valid_coords
0,San Francisco Caltrain Station 2 (Townsend St...,3394,4621,37.776639,-122.395526,8015,1227,15.308796,True
1,Market St at 10th St,3648,3702,37.776619,-122.417385,7350,54,0.734694,True
2,Montgomery St BART Station (Market St at 2nd St),2707,3458,37.789625,-122.400811,6165,751,12.181671,True
3,Berry St at 4th St,2951,2770,37.775880,-122.393170,5721,-181,-3.163783,True
4,San Francisco Ferry Building (Harry Bridges Pl...,2539,3151,37.795392,-122.394203,5690,612,10.755712,True
5,Powell St BART Station (Market St at 4th St),2620,2852,37.786375,-122.404904,5472,232,4.239766,True
6,San Francisco Caltrain (Townsend St at 4th St),2569,2860,37.776598,-122.395282,5429,291,5.360103,True
7,Steuart St at Market St,2181,2262,37.794130,-122.394430,4443,81,1.823093,True
8,The Embarcadero at Sansome St,1975,2341,37.804770,-122.403234,4316,366,8.480074,True
9,Powell St BART Station (Market St at 5th St),2143,2152,37.783899,-122.408445,4295,9,0.209546,True


In [18]:
from pathlib import Path

print(Path.cwd())


c:\DiskD\AI\Depi-Projects-AI-upload\Data-Analysis\ford_gobike_analysis\Gold_DF\station_metrics


In [ ]:
from pathlib import Path

output_path = Path("Gold_DF/station_metrics/Station_Metrics.csv")

output_path.parent.mkdir(parents=True, exist_ok=True)

station_metrics.to_csv(
    output_path,
    index=False
)

In [19]:
station_metrics.columns.tolist()

['station_name',
 'departures',
 'arrivals',
 'lat',
 'lon',
 'total_traffic',
 'net_flow',
 'imbalance_ratio',
 'has_valid_coords']